# GAS-BayesSHAP — audit-fix reruns

Reruns the experiments that the latest deep audit (commit `af2d62e`) invalidated or found incomplete.  Orchestrates the real CLI scripts only; duplicates no scientific algorithm.  Each section states what changed and why the rerun is needed.

## 0. Environment & config

In [13]:
import sys, os, time, json, subprocess
from pathlib import Path
sys.path.insert(0, "..")
import gas_bayesshap
ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
print("GAS-BayesSHAP", gas_bayesshap.__version__)

SOTA_N   = int(os.environ.get("SOTA_N", "20"))
FP_N     = int(os.environ.get("FP_N", "50"))
REG_N    = int(os.environ.get("REG_N", "20"))
EPS      = float(os.environ.get("GAS_EPS", "0.05"))
BUDGET   = int(os.environ.get("GAS_BUDGET", "3000"))
SKIP     = set(os.environ.get("GAS_SKIP", "").split(",")) - {""}

def run(*args, tag="", skip=False):
    if skip:
        print(f"--- SKIPPED: {tag or ' '.join(args)}"); return 0.0
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT)
    dt = time.time() - t0
    if r.returncode != 0:
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"SOTA_N={SOTA_N} FP_N={FP_N} REG_N={REG_N} SKIP={sorted(SKIP)}")

GAS-BayesSHAP 11.0.0
SOTA_N=20 FP_N=50 REG_N=20 SKIP=[]


## A. SOTA-style baselines — FIXED GP design (audit P0-1)

**Bug:** `run_sota_baselines.py` re-seeded `RandomState(7 + i)` inside the design loop, so all 256 design masks were identical (the committed `paper_sota_baselines_comparison.csv` shows a constant ShaplEIG-style RMSE across K for each instance).  **Fix:** one rng per instance advanced per draw, design deduplicated and sized from K, outputs renamed to `paper_reference_baselines_ablation.csv` (OddSHAP-style = log-odds transform reference, GP-quadrature = surrogate reference; **not** a matched-budget comparison).

In [ ]:
run("run_sota_baselines.py", "--n", str(SOTA_N),
    tag=f"A. reference baselines (fixed design) N={SOTA_N}", skip="A" in SKIP)

In [ ]:
import pandas as pd
p = ROOT / "main_results" / "paper_reference_baselines_ablation.csv"
if p.exists():
    d = pd.read_csv(p)
    print(d.groupby(["dataset", "K"]).agg(
        gas=("gas_rmse", "mean"),
        odd_logodds=("odd_logodds_rmse", "mean"),
        gp_quadrature=("gp_quadrature_rmse", "mean"),
        gp_unique=("gp_unique_coalitions", "min"),
    ).round(5).to_string())
    # sanity: GP design must now contain distinct coalitions
    print("\nmin unique coalitions across runs:", int(d["gp_unique_coalitions"].min()))
    assert d["gp_unique_coalitions"].min() > 1, "design still degenerate!"

## B. N=50 finite-population with certificate diagnostics (audit P0-4)

The committed fp N=50 CSVs predate the diagnostics fields.  The runner now records `delta1_coupon`, `reported_coverage_level`, `coupon_threshold_satisfied`, `certificate_at_nominal_level`, `certificate_is_rigorous` per instance and aggregates them in the summary, so `simultaneous_coverage_rate=1.0` is interpretable as an empirical frequency alongside the formal certificate status.  ~2.5–3 h for both datasets.

In [ ]:
run("run_paper_experiments.py", "--only", "wine", "--n", str(FP_N),
    "--eps", str(EPS), "--budget", str(BUDGET), "--range-mode", "finite_population",
    tag=f"B1. wine fp N={FP_N} + diagnostics", skip="B" in SKIP)

In [ ]:
run("run_paper_experiments.py", "--only", "air", "--n", str(FP_N),
    "--eps", str(EPS), "--budget", str(BUDGET), "--range-mode", "finite_population",
    tag=f"B2. air fp N={FP_N} + diagnostics", skip="B" in SKIP)

## C. Matched-budget curves with wall-clock (audit P0-3)

The committed curves already carry `*_evals_actual` columns; the runner now also records per-method wall-clock seconds (`gas_wall_s`, `kernel_wall_s`, `mc_wall_s`) so the nominal-vs-actual gap is fully quantified.  ~1 h for both datasets.

In [19]:
run("run_paper_experiments.py", "--only", "curves", "--range-mode", "spec",
    tag="C. instrumented curves + wall-clock", skip="C" in SKIP)


>>> C. instrumented curves + wall-clock
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000357 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000309 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
  K=128

5811.292855978012

## D. Regime semantics — duplicate-name suffixing (audit P1-8)

`name_regime` mapped two k-means clusters to `clean_air`, silently collapsing distinct subregimes.  The script now suffixes duplicates (`clean_air_2`) and prints the full mapping.  Run with N=20 (default) or N=60 for ≥20 instances per named regime.

In [14]:
run("regime_semantics.py", "--n", str(REG_N), "--clusters", "4",
    "--eps", str(EPS), "--budget", str(BUDGET),
    tag=f"D. regime semantics N={REG_N} (suffixed names)", skip="D" in SKIP)


>>> D. regime semantics N=20 (suffixed names)
named regimes: {0: 'winter_smog', 1: 'photochemical', 2: 'clean_air', 3: 'clean_air_2'}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000467 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2163
[LightGBM] [Info] Number of data points in the train set: 22313, number of used features: 11
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
  inst 0 [winter_smog   ] rmse=0.00097 rho=0.642 top3=PM10,CO,PM2.5
  inst 1 [photochemical ] rmse=0.00132 rho=0.536 top3=O3,NO2,WSPM
  inst 2 [clean_air     ] rmse=0.00285 rho=0.400 top3=WSPM,DEWP,O3
  inst 3 [clean_air_2   ] rmse=0.00129 rho=-0.075 top3=DEWP,WSPM,PRES
  inst 4 [winter_smog   ] rmse=0.00106 rho=0.586 top3=CO,PM10,SO2
  inst 5 [photochemical

656.3536660671234

## E. Finite-population Tier-B (audit P1-9)

Tier-B (group-lag, M=66 → 11 macros) has only been run with the spec range.  This runs it with `range_mode=finite_population` to check whether group-level certification improves at the same budget.

In [15]:
run("run_paper_experiments.py", "--only", "tierb", "--n", "20",
    "--eps", str(EPS), "--budget", str(BUDGET), "--range-mode", "finite_population",
    tag="E. Tier-B finite-population N=20", skip="E" in SKIP)


>>> E. Tier-B finite-population N=20
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001399 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12937
[LightGBM] [Info] Number of data points in the train set: 22296, number of used features: 66
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[tierB] lagged surrogate acc=0.9749
[tierB] summary: rmse=0.00195 sim_cov=1.00 sign_cert=0.000 mean_width=1.61

done in 1632s; results in results/paper_experiments/ and main_results/paper_*.csv
<<< done in 27.2 min


1633.8271698951721

## F. Reproduction manifest (audit P1-10)

In [16]:
run("reproduce_paper.py", "--manifest",
    tag="F. write reproduce_manifest.json", skip="F" in SKIP)


>>> F. write reproduce_manifest.json
manifest written to main_results/reproduce_manifest.json (commit 31222aef81cda03440c7b097802b780e286fecfb)
<<< done in 0.0 min


0.12099409103393555

In [17]:
m = json.loads((ROOT / "main_results" / "reproduce_manifest.json").read_text())
print("commit:", m["commit"], "| dirty:", m["git_dirty"],
      "| artifacts hashed:", m["n_artifacts"])
print("\nTo regenerate everything from scratch:\n"
      "  python scripts/reproduce_paper.py --all")

commit: 31222aef81cda03440c7b097802b780e286fecfb | dirty: True | artifacts hashed: 45

To regenerate everything from scratch:
  python scripts/reproduce_paper.py --all


## Expected runtime (laptop) and notes
- **Full run ≈ 5–6 h:** A ≈ 40–60 min, B ≈ 2.5–3 h, C ≈ 1 h,
  D ≈ 15 min, E ≈ 35 min, F ≈ 0.
- **Smoke run (≈ 12 min):** `GAS_SKIP=B,C,E SOTA_N=2 REG_N=4`
- **What each section can/cannot claim is stated in its markdown cell.**
  In particular: A replaces the invalid ShaplEIG artifact entirely;
  B does not change any RMSE/coverage numbers (same seeds), it adds
  certificate diagnostics; C adds wall-clock to the existing curves.

In [18]:
!python ../scripts/run_paper_experiments.py --only curves

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000283 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
  K=128: gas=0.00430 kernel=0.00505 mc=0.04330 |